# 05 — Spectra Inspector

Interactive viewer for Aeris spectra during the interesting periods listed in `interesting_periods.txt`.

**Layout (per load):**
1. Gas concentrations (CH4, C2H6, C3H8) from merged/calibrated data — event period shaded
2. Spectra heatmap — time on X, channel 1–1024 on Y, color = Δ% from pre-event background
3. Spectrum overlay — background mean vs event mean; click heatmap to add a single timestamp
4. Difference spectrum — (event − background) / |background| × 100%

**Loading strategy:** Ultra321/Pico017 files are ~670 MB each (1 Hz × 24 h × 1024 channels).
A two-pass reader is used: read TIMESTAMP-only to find matching rows, then re-read only those rows
with full columns (~6 s per file via OS page cache on the second pass).
A file-range index is built once and cached to disk so subsequent notebook starts are instant.

**To rebuild the file index** (e.g. after new files are added):
```python
_FILE_INDEX = _build_file_index(force=True)
```

In [ ]:
import json
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
_BASE         = Path('/uufs/chpc.utah.edu/common/home/lin-group24/agm/Mobile_SLV/Data/2026')
_RECLEANED    = _BASE / 'recleaned'
_CAL_DIR      = _BASE / 'calibrated'
_MERGED_DIR   = _BASE / 'merged'
_GAS_DIR      = _CAL_DIR if _CAL_DIR.exists() else _MERGED_DIR
_PERIODS_FILE = Path('/uufs/chpc.utah.edu/common/home/u0890904/LAIR_1/Projects/Mobile_SLV/Data/interesting_periods.txt')
_INDEX_CACHE  = _RECLEANED / '.spectra_index_cache.json'

_SPECTRA_DIRS = {
    'Ultra321': _RECLEANED / 'LANL_aerisultra321' / 'Spectra',
    'Pico017':  _RECLEANED / 'LANL_aerispico017'  / 'Spectra',
    'Ultra460': _RECLEANED / 'WYO_aerisultra460'   / 'Spectralite',
}
_INST_HZ     = {'Ultra321': '~1 Hz', 'Pico017': '~1 Hz', 'Ultra460': '~1/min'}
_INST_COLORS = {'Ultra321': '#2ca02c', 'Pico017': '#d62728', 'Ultra460': '#ff7f0e'}
_GAS_COLORS  = {
    'CH4': '#1f77b4', 'C2H6': '#2ca02c', 'C3H8': '#d62728',
    'C3H6': '#9467bd', 'CO2': '#7f7f7f', 'CO': '#bcbd22',
}

print(f'Gas context dir : {_GAS_DIR}')
for inst, d in _SPECTRA_DIRS.items():
    n = len(list(d.glob('*_clean.csv')))
    print(f'  {inst:10s}: {n} files')

In [ ]:
# ── Interesting periods ────────────────────────────────────────────────────────
def _parse_periods():
    # skipinitialspace=True handles the "dt_start, dt_end, description" header spacing
    df = pd.read_csv(_PERIODS_FILE, skipinitialspace=True)
    for col in ('dt_start', 'dt_end'):
        df[col] = pd.to_datetime(df[col], utc=True).dt.tz_convert(None)  # naive UTC
    return df[~df['description'].str.contains('calibration', case=False)].reset_index(drop=True)

_PERIODS = _parse_periods()
print(f'Loaded {len(_PERIODS)} non-calibration periods:')
for _, r in _PERIODS.iterrows():
    dur = int((r.dt_end - r.dt_start).total_seconds())
    print(f"  {r.dt_start.strftime('%Y-%m-%d %H:%M')}–{r.dt_end.strftime('%H:%M')} UTC  "
          f"({dur}s)  {r.description}")

# ── File index (timestamp range per spectra file, cached to disk) ──────────────
def _build_file_index(force=False):
    """Scan TIMESTAMP min/max for every spectra file. Result cached to _INDEX_CACHE."""
    if not force and _INDEX_CACHE.exists():
        data = json.loads(_INDEX_CACHE.read_text())
        index = {}
        for inst, entries in data.items():
            index[inst] = [
                (Path(p), pd.Timestamp(lo), pd.Timestamp(hi))
                for p, lo, hi in entries
            ]
        total = sum(len(v) for v in index.values())
        print(f'Loaded file index from cache ({total} files). '
              f'Delete {_INDEX_CACHE.name} or call _build_file_index(force=True) to rebuild.')
        return index

    print('Building spectra file index (first run, ~2 min, cached after)...')
    index = {}
    raw   = {}
    for inst, d in _SPECTRA_DIRS.items():
        files = sorted(d.glob('*_clean.csv'))
        entries = []
        for i, f in enumerate(files, 1):
            try:
                ts = pd.read_csv(f, usecols=['TIMESTAMP'],
                                 parse_dates=['TIMESTAMP'])['TIMESTAMP']
                lo, hi = ts.min(), ts.max()
                entries.append((f, lo, hi))
                print(f'  {inst} {i:2d}/{len(files)}: {f.name}  [{lo} – {hi}]')
            except Exception as e:
                print(f'  {inst}: skip {f.name}: {e}')
        index[inst] = entries
        raw[inst]   = [(str(f), str(lo), str(hi)) for f, lo, hi in entries]

    _INDEX_CACHE.write_text(json.dumps(raw))
    print(f'Index cached → {_INDEX_CACHE}')
    return index

_FILE_INDEX = _build_file_index()

In [ ]:
# ── Spectra loader (two-pass for large files) ──────────────────────────────────
def load_spectra(inst, t_start, t_end):
    """
    Load spectra rows within [t_start, t_end] (naive UTC timestamps).

    Two-pass strategy for the ~670 MB Ultra321/Pico017 daily files:
      Pass 1: read TIMESTAMP column only (~4 s) to find matching row indices.
      Pass 2: re-read the file skipping all other rows (~3 s from OS cache).
    Ultra460 spectralite files are small (~5 MB) so one pass suffices.

    Returns DataFrame with spec_0001…spec_1024 columns indexed by TIMESTAMP, or None.
    """
    candidates = [
        (f, lo, hi) for f, lo, hi in _FILE_INDEX[inst]
        if hi >= t_start and lo <= t_end
    ]
    if not candidates:
        return None

    chunks = []
    for f, _, _ in candidates:
        # Pass 1 — timestamps only
        ts = pd.read_csv(f, usecols=['TIMESTAMP'],
                         parse_dates=['TIMESTAMP'])['TIMESTAMP']
        row_mask = (ts >= t_start) & (ts <= t_end)
        if not row_mask.any():
            continue
        keep = frozenset(row_mask[row_mask].index + 1)  # 1-based (row 0 = header line)

        # Pass 2 — full columns, matching rows only
        df = pd.read_csv(
            f, index_col='TIMESTAMP', parse_dates=True,
            skiprows=lambda i, _k=keep: i != 0 and i not in _k,
        )
        spec_cols = [c for c in df.columns if c.startswith('spec_')]
        chunks.append(df[spec_cols])

    return pd.concat(chunks).sort_index() if chunks else None


# ── Gas context loader ─────────────────────────────────────────────────────────
def load_gas(t_start, t_end):
    """Load merged/calibrated gas data for [t_start, t_end] (naive UTC). Returns DataFrame or None."""
    tags = set()
    cur = pd.Timestamp(t_start.date())
    while cur <= pd.Timestamp(t_end.date()) + pd.Timedelta('1D'):
        tags.add(cur.strftime('%Y%m%d'))
        cur += pd.Timedelta('1D')

    frames = []
    for tag in sorted(tags):
        path = _GAS_DIR / f'{tag}.csv'
        if not path.exists():
            continue
        df = pd.read_csv(path)
        df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'])
        mask = (df['TIMESTAMP'] >= t_start) & (df['TIMESTAMP'] <= t_end)
        if mask.any():
            frames.append(df[mask])

    return (
        pd.concat(frames).sort_values('TIMESTAMP').reset_index(drop=True)
        if frames else None
    )

In [ ]:
# ── Figure builders ────────────────────────────────────────────────────────────

def _gas_cols_for(gas_df, inst, gases=('CH4', 'C2H6', 'C3H8', 'C3H6')):
    """Return {gas: best_col} preferring inst-specific raw (no _cal) columns."""
    result = {}
    for gas in gases:
        raw_cols  = [c for c in gas_df.columns
                     if c.startswith(gas + '_') and '_cal' not in c]
        inst_cols = [c for c in raw_cols if inst in c]
        col = (inst_cols or raw_cols or [None])[0]
        if col:
            result[gas] = col
    return result


def build_gas_fig(gas_df, t0, t1, inst, context_min):
    """Gas time-series with event shaded. Prefers inst-specific columns."""
    if gas_df is None or gas_df.empty:
        fig = go.Figure()
        fig.update_layout(height=180, template='plotly_white',
                          annotations=[dict(text='No gas data', showarrow=False)])
        return fig

    cols = _gas_cols_for(gas_df, inst)
    ctx  = pd.Timedelta(minutes=context_min)
    fig  = go.Figure()

    for gas, col in cols.items():
        valid = gas_df['TIMESTAMP'].notna() & gas_df[col].notna()
        unit  = col.split('_')[1] if '_' in col else ''
        fig.add_trace(go.Scattergl(
            x=gas_df.loc[valid, 'TIMESTAMP'],
            y=gas_df.loc[valid, col],
            name=f'{gas} ({unit})  [{col.split("_")[-1]}]',
            line=dict(color=_GAS_COLORS.get(gas, '#888'), width=1.5),
            hovertemplate='%{x|%H:%M:%S}<br>%{y:.3f}<extra>%{fullData.name}</extra>',
        ))

    fig.add_vrect(x0=t0, x1=t1, fillcolor='rgba(255,165,0,0.2)',
                  layer='below', line_width=0)
    fig.update_layout(
        height=200, template='plotly_white', hovermode='x unified',
        margin=dict(l=65, r=180, t=30, b=5),
        xaxis=dict(showticklabels=False, range=[t0 - ctx, t1 + ctx]),
        yaxis=dict(title='Concentration', title_font_size=10),
        legend=dict(x=1.01, y=1, font_size=9,
                    bgcolor='rgba(255,255,255,0.8)', bordercolor='#ccc', borderwidth=1),
        title=dict(text=f'Gas concentrations — {inst} ({_INST_HZ[inst]})', font_size=11),
    )
    return fig


def build_heatmap_fig(spec_df, t0, t1, inst, normalize=True, clip_pct=2.0):
    """
    Spectra heatmap: time on X, channel 1-1024 on Y.

    normalize=True  → color = Δ% from pre-event background mean (diverging colorscale).
    normalize=False → color = raw intensity (Viridis).
    clip_pct: percentile used for symmetric colorscale limits (avoids outlier domination).

    Returns a FigureWidget so on_click can be attached by the caller.
    """
    if spec_df is None or spec_df.empty:
        fw = go.FigureWidget()
        fw.update_layout(height=340, template='plotly_white',
                         annotations=[dict(text=f'No spectra for {inst}',
                                          showarrow=False, font_size=14)])
        return fw

    times = spec_df.index
    Z     = spec_df.values.astype(float)   # (n_times, n_channels)
    n_ch  = Z.shape[1]
    channels = np.arange(1, n_ch + 1)

    if normalize:
        bg_mask = times < t0
        bg = Z[bg_mask].mean(axis=0) if bg_mask.any() else Z.mean(axis=0)
        bg = np.where(np.abs(bg) < 1e-10, np.nan, bg)
        Z_plot = (Z - bg) / np.abs(bg) * 100.0
        finite = Z_plot[np.isfinite(Z_plot)]
        clim   = float(np.percentile(np.abs(finite), 100 - clip_pct)) if finite.size else 1.0
        hm_kw  = dict(
            colorscale='RdBu_r', zmid=0, zmin=-clim, zmax=clim,
            colorbar=dict(
                title=dict(text='Δ%', font=dict(size=10)),
                thickness=14, len=0.85,
            ),
        )
        title_label = f'{inst} spectra — normalized (Δ% from pre-event bg)'
    else:
        Z_plot = Z
        hm_kw  = dict(
            colorscale='Viridis',
            colorbar=dict(
                title=dict(text='Intensity', font=dict(size=10)),
                thickness=14, len=0.85,
            ),
        )
        title_label = f'{inst} spectra — raw intensity'

    fw = go.FigureWidget(go.Heatmap(
        x=times,
        y=channels,
        z=Z_plot.T,          # shape: (n_channels, n_times) — channels on Y axis
        hovertemplate='t=%{x|%H:%M:%S.%2f}<br>ch=%{y}<br>val=%{z:.2f}<extra></extra>',
        name='spectra',
        **hm_kw,
    ))
    fw.add_vrect(x0=t0, x1=t1, fillcolor='rgba(255,165,0,0.12)',
                 layer='below', line_width=0)
    fw.update_layout(
        height=360, template='plotly_white',
        margin=dict(l=65, r=130, t=30, b=50),
        xaxis=dict(tickformat='%H:%M:%S', title='UTC'),
        yaxis=dict(title='Channel', autorange=True, tickfont_size=9),
        title=dict(text=title_label, font_size=11),
    )
    return fw


def build_spectrum_fig(spec_df, t0, t1, selected_time=None):
    """
    Overlay: background mean (dashed grey), event mean (red), selected timestamp (blue dotted).
    Returns FigureWidget — caller may re-call with a selected_time to add a trace.
    """
    fw = go.FigureWidget()
    if spec_df is None or spec_df.empty:
        return fw

    times    = spec_df.index
    Z        = spec_df.values.astype(float)
    channels = np.arange(1, Z.shape[1] + 1)
    bg_mask  = times < t0
    ev_mask  = (times >= t0) & (times <= t1)

    if bg_mask.any():
        fw.add_trace(go.Scattergl(
            x=channels, y=Z[bg_mask].mean(axis=0),
            name=f'Background mean (n={bg_mask.sum()})',
            line=dict(color='#555', width=1.5, dash='dash'),
            hovertemplate='ch=%{x}<br>%{y:.0f}<extra>Background</extra>',
        ))
    if ev_mask.any():
        fw.add_trace(go.Scattergl(
            x=channels, y=Z[ev_mask].mean(axis=0),
            name=f'Event mean (n={ev_mask.sum()})',
            line=dict(color='#e15759', width=2),
            hovertemplate='ch=%{x}<br>%{y:.0f}<extra>Event mean</extra>',
        ))
    if selected_time is not None:
        if hasattr(selected_time, 'tzinfo') and selected_time.tzinfo is not None:
            selected_time = selected_time.tz_convert(None)
        idx    = int(np.argmin(np.abs(times - selected_time)))
        ts_str = times[idx].strftime('%H:%M:%S')
        fw.add_trace(go.Scattergl(
            x=channels, y=Z[idx],
            name=f'Selected {ts_str}',
            line=dict(color='#4e79a7', width=1.2, dash='dot'),
            hovertemplate='ch=%{x}<br>%{y:.0f}<extra>Selected</extra>',
        ))

    fw.update_layout(
        height=280, template='plotly_white', hovermode='x unified',
        margin=dict(l=65, r=20, t=30, b=50),
        xaxis=dict(title='Channel'),
        yaxis=dict(title='Intensity (raw counts)'),
        legend=dict(x=0.01, y=0.99, font_size=9,
                    bgcolor='rgba(255,255,255,0.8)', bordercolor='#ccc', borderwidth=1),
        title=dict(text='Spectra overlay — click heatmap (or use text box below) to add a selected timestamp',
                   font_size=11),
    )
    return fw


def build_diff_fig(spec_df, t0, t1):
    """Difference spectrum: (event_mean − bg_mean) / |bg_mean| × 100 %."""
    fw = go.FigureWidget()
    if spec_df is None or spec_df.empty:
        return fw

    times    = spec_df.index
    Z        = spec_df.values.astype(float)
    channels = np.arange(1, Z.shape[1] + 1)
    bg_mask  = times < t0
    ev_mask  = (times >= t0) & (times <= t1)

    if not bg_mask.any() or not ev_mask.any():
        fw.update_layout(height=230, annotations=[
            dict(text='Need pre-event context to compute difference spectrum',
                 showarrow=False, font_size=12)])
        return fw

    bg_mean = Z[bg_mask].mean(axis=0)
    ev_mean = Z[ev_mask].mean(axis=0)
    denom   = np.where(np.abs(bg_mean) < 1.0, 1.0, np.abs(bg_mean))
    diff    = (ev_mean - bg_mean) / denom * 100.0

    fw.add_trace(go.Scattergl(
        x=channels, y=diff,
        line=dict(color='#e15759', width=1),
        fill='tozeroy', fillcolor='rgba(225,87,89,0.12)',
        name='Δ%',
        hovertemplate='ch=%{x}<br>Δ%%=%{y:.3f}<extra></extra>',
    ))
    fw.add_hline(y=0, line=dict(color='#555', width=1, dash='dash'))
    fw.update_layout(
        height=240, template='plotly_white',
        margin=dict(l=65, r=20, t=30, b=50),
        xaxis=dict(title='Channel'),
        yaxis=dict(title='Δ% from background'),
        title=dict(text='Difference spectrum: (event − background) / |background| × 100%',
                   font_size=11),
    )
    return fw

In [ ]:
# ── Interactive widget ─────────────────────────────────────────────────────────

def _period_label(row):
    ds  = row.dt_start.strftime('%b-%d %H:%M')
    de  = row.dt_end.strftime('%H:%M')
    dur = int((row.dt_end - row.dt_start).total_seconds())
    return f'{ds}–{de} UTC ({dur}s)  |  {row.description}'

period_opts = [(_period_label(r), i) for i, r in _PERIODS.iterrows()]

# ── Controls ──────────────────────────────────────────────────────────────────
period_dd = widgets.Dropdown(
    options=period_opts, value=0,
    description='Period:',
    style={'description_width': '55px'},
    layout=widgets.Layout(width='720px'),
)
inst_dd = widgets.Dropdown(
    options=['Ultra321', 'Pico017', 'Ultra460'],
    value='Ultra321', description='Instr:',
    style={'description_width': '42px'},
    layout=widgets.Layout(width='180px'),
)
ctx_slider = widgets.IntSlider(
    value=3, min=1, max=15, step=1,
    description='Context (min):',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='330px'),
)
norm_cb = widgets.Checkbox(
    value=True, description='Normalize (Δ% from bg)', indent=False,
    layout=widgets.Layout(width='215px'),
)
load_btn = widgets.Button(
    description='Load & Plot', button_style='primary',
    icon='search', layout=widgets.Layout(width='130px', height='34px'),
)

# Timestamp text selector — fallback when FigureWidget on_click isn't available,
# or for precise timestamp entry. Accepts HH:MM:SS (uses event date) or full datetime.
ts_input = widgets.Text(
    value='', placeholder='HH:MM:SS  or  YYYY-MM-DD HH:MM:SS',
    description='Pick time:',
    style={'description_width': '72px'},
    layout=widgets.Layout(width='340px'),
)
ts_btn = widgets.Button(
    description='Show spectrum', button_style='info',
    layout=widgets.Layout(width='135px', height='34px'),
)

status_lbl = widgets.HTML(
    '<span style="color:#888;font-size:11px;">Select a period and click Load &amp; Plot.</span>')

out_gas  = widgets.Output()
out_heat = widgets.Output()
out_spec = widgets.Output()
out_diff = widgets.Output()

_ctx = {}   # shared mutable state

# ── Spectrum update helper ─────────────────────────────────────────────────────
def _show_spectrum_at(clicked_ts):
    spec_df = _ctx.get('spec_df')
    if spec_df is None:
        return
    with out_spec:
        clear_output(wait=True)
        display(build_spectrum_fig(spec_df, _ctx['t0'], _ctx['t1'],
                                   selected_time=clicked_ts))

# ── Heatmap click → update spectrum panel ─────────────────────────────────────
def _on_heatmap_click(trace, points, selector):
    if not points.xs:
        return
    try:
        clicked = pd.Timestamp(points.xs[0])
        if clicked.tzinfo is not None:
            clicked = clicked.tz_convert(None)
    except Exception:
        return
    _show_spectrum_at(clicked)

# ── Text-box timestamp selector ────────────────────────────────────────────────
def _on_ts_btn(_):
    raw = ts_input.value.strip()
    if not raw:
        return
    try:
        if re.match(r'^\d{2}:\d{2}:\d{2}', raw):
            date_str = _ctx.get('t0', pd.Timestamp('2026-01-01')).strftime('%Y-%m-%d')
            clicked = pd.Timestamp(f'{date_str} {raw}')
        else:
            clicked = pd.Timestamp(raw)
    except Exception as e:
        status_lbl.value = f'<span style="color:red;font-size:11px;">Bad timestamp: {e}</span>'
        return
    _show_spectrum_at(clicked)

ts_btn.on_click(_on_ts_btn)

# ── Main load callback ────────────────────────────────────────────────────────
def _on_load(_):
    row  = _PERIODS.iloc[period_dd.value]
    inst = inst_dd.value
    ctx  = ctx_slider.value
    norm = norm_cb.value

    t0   = row.dt_start
    t1   = row.dt_end
    t_lo = t0 - pd.Timedelta(minutes=ctx)
    t_hi = t1 + pd.Timedelta(minutes=ctx)

    status_lbl.value = (
        f'<span style="color:#e67e00;font-size:11px;">'  
        f'Loading {inst} spectra and gas data…</span>'
    )

    spec_df = load_spectra(inst, t_lo, t_hi)
    gas_df  = load_gas(t_lo, t_hi)
    _ctx.update({'spec_df': spec_df, 't0': t0, 't1': t1, 'inst': inst})

    with out_gas:
        clear_output(wait=True)
        display(build_gas_fig(gas_df, t0, t1, inst, ctx))

    with out_heat:
        clear_output(wait=True)
        if spec_df is not None:
            fw = build_heatmap_fig(spec_df, t0, t1, inst, normalize=norm)
            fw.data[0].on_click(_on_heatmap_click)
            display(fw)
        else:
            print(f'No {inst} spectra files found covering this period.')

    with out_spec:
        clear_output(wait=True)
        display(build_spectrum_fig(spec_df, t0, t1))

    with out_diff:
        clear_output(wait=True)
        display(build_diff_fig(spec_df, t0, t1))

    n_sp = len(spec_df) if spec_df is not None else 0
    n_gd = len(gas_df)  if gas_df  is not None else 0
    status_lbl.value = (
        f'<span style="color:#27ae60;font-size:11px;">'  
        f'Loaded {n_sp} spectra rows, {n_gd} gas rows. '
        f'Click the heatmap to overlay a single timestamp, or type a time below.'  
        f'</span>'
    )

load_btn.on_click(_on_load)

# ── Layout ────────────────────────────────────────────────────────────────────
ctrl = widgets.VBox([
    widgets.HBox([period_dd, inst_dd]),
    widgets.HBox([ctx_slider, norm_cb, load_btn]),
    widgets.HBox([ts_input, ts_btn]),
    status_lbl,
], layout=widgets.Layout(
    border='1px solid #ccc', padding='10px 14px',
    background_color='#fafafa', margin='0 0 12px 0',
))

display(ctrl, out_gas, out_heat, out_spec, out_diff)

## Two-period comparison

Compare difference spectra for two periods side-by-side — useful for distinguishing real C2H6/C3H8 signals
from spectral cross-talk artifacts (e.g. a *coherent* spike vs an *offset* spike from CH4).

Edit `PERIOD_A`, `PERIOD_B`, `INST`, and `CTX_MIN` below and run the cell.

---
## Simple Spectra Viewer

Gas concentrations panel (with cursor tracking position) above a scrubber that steps
through every individual background-normalised spectrum (I/I₀) in the loaded window.
The spectrum title shows reported gas concentrations at that moment.
Spectra are coloured grey (pre/post event) → blue→red (through event).

In [ ]:
# ── Controls ──────────────────────────────────────────────────────────────────
sv_period_dd = widgets.Dropdown(
    options=period_opts, value=0,
    description='Period:', style={'description_width': '55px'},
    layout=widgets.Layout(width='720px'),
)
sv_inst_dd = widgets.Dropdown(
    options=['Ultra321', 'Pico017', 'Ultra460'], value='Ultra321',
    description='Instr:', style={'description_width': '42px'},
    layout=widgets.Layout(width='180px'),
)
sv_ctx_slider = widgets.IntSlider(
    value=3, min=1, max=15, step=1,
    description='Context (min):', style={'description_width': '100px'},
    layout=widgets.Layout(width='300px'),
)
sv_load_btn = widgets.Button(
    description='Load', button_style='primary',
    icon='search', layout=widgets.Layout(width='90px', height='34px'),
)
sv_status = widgets.HTML(
    '<span style="color:#888;font-size:11px;">Select a period and click Load.</span>')

sv_slider = widgets.IntSlider(
    value=0, min=0, max=1, step=1,
    description='', continuous_update=True,
    layout=widgets.Layout(width='98%'),
    style={'handle_color': '#4e79a7'},
)
sv_time_lbl = widgets.HTML('<span style="font-size:11px;color:#555;">—</span>')

def _sv_placeholder(height):
    fw = go.FigureWidget()
    fw.update_layout(height=height, template='plotly_white',
                     margin=dict(l=65, r=20, t=38, b=50),
                     annotations=[dict(text='Select a period and click Load.',
                                       showarrow=False, font_size=11)])
    return fw

_sv_fw_gas = _sv_placeholder(190)
_sv_fw_sel = _sv_placeholder(290)

_sv = {}


# ── Helpers ────────────────────────────────────────────────────────────────────

def _sv_baseline_correct(Z):
    """Per-spectrum upper-envelope baseline correction (removes laser ramp).

    Iteratively smooths each spectrum while lifting absorption dips up to the
    smooth level each pass, so the fitted curve tracks the upper envelope.
    Dividing by the result gives I/baseline where absorptions dip below 1.0.
    """
    from scipy.signal import savgol_filter
    n_ch = Z.shape[1]
    # Window ~20% of channels, must be odd and >= 5
    win = max(5, (n_ch // 5) | 1)

    Z_up = Z.astype(float).copy()
    for _ in range(8):
        smooth = savgol_filter(Z_up, win, polyorder=3, axis=1, mode='nearest')
        Z_up   = np.maximum(Z, smooth)   # lift dips up to smooth level
    baseline = savgol_filter(Z_up, win, polyorder=3, axis=1, mode='nearest')

    denom = np.where(baseline < 1.0, np.nan, baseline)
    return Z / denom


def _sv_spectrum_color(ts, t0, t1):
    if ts < t0:  return '#aaaaaa'
    if ts > t1:  return '#888888'
    f = (ts - t0).total_seconds() / max((t1 - t0).total_seconds(), 1)
    return f'rgb({int(70+165*f)},{int(130-100*f)},{int(180-120*f)})'


def _sv_gas_str(gas_df, ts, inst):
    if gas_df is None or gas_df.empty:
        return ''
    row = gas_df.iloc[(gas_df['TIMESTAMP'] - ts).abs().argmin()]
    parts = []
    for gas in ('CH4', 'C2H6', 'C3H8', 'C3H6'):
        raw = [c for c in gas_df.columns
               if c.startswith(gas + '_') and '_cal' not in c and inst in c]
        if not raw:
            raw = [c for c in gas_df.columns
                   if c.startswith(gas + '_') and '_cal' not in c]
        if raw and pd.notna(row.get(raw[0])):
            parts.append(f'{gas}={row[raw[0]]:.3f} {raw[0].split("_")[1]}')
    return '  '.join(parts)


# ── In-place figure updaters ───────────────────────────────────────────────────

def _sv_load_gas(fw, gas_df, t0, t1, inst, ctx_min):
    ctx_td = pd.Timedelta(minutes=ctx_min)
    fw.data = []
    fw.update_layout(annotations=[])
    if gas_df is not None and not gas_df.empty:
        cols = _gas_cols_for(gas_df, inst)
        for gas, col in cols.items():
            valid = gas_df['TIMESTAMP'].notna() & gas_df[col].notna()
            unit  = col.split('_')[1] if '_' in col else ''
            fw.add_trace(go.Scattergl(
                x=gas_df.loc[valid, 'TIMESTAMP'], y=gas_df.loc[valid, col],
                name=f'{gas} ({unit})',
                line=dict(color=_GAS_COLORS.get(gas, '#888'), width=1.5),
                hovertemplate='%{x|%H:%M:%S}<br>%{y:.3f}<extra>%{fullData.name}</extra>',
            ))
    fw.update_layout(
        shapes=[
            dict(type='rect', x0=t0, x1=t1, y0=0, y1=1, xref='x', yref='paper',
                 fillcolor='rgba(255,165,0,0.15)', layer='below', line=dict(width=0)),
            dict(type='line', x0=t0.isoformat(), x1=t0.isoformat(),
                 y0=0, y1=1, xref='x', yref='paper',
                 line=dict(color='#222', width=1.5, dash='dot')),
        ],
        height=190, template='plotly_white', hovermode='x unified',
        margin=dict(l=65, r=20, t=30, b=5),
        xaxis=dict(showticklabels=False, range=[t0 - ctx_td, t1 + ctx_td]),
        yaxis=dict(title='Concentration', title_font_size=10),
        legend=dict(x=1.01, y=1, font_size=9,
                    bgcolor='rgba(255,255,255,0.8)', bordercolor='#ccc', borderwidth=1),
        title=dict(text=f'{inst} — gas concentrations', font_size=11),
    )


def _sv_load_sel(fw, Z_norm, channels, idx, t0, t1, times):
    color = _sv_spectrum_color(times[idx], t0, t1)
    fw.data = []
    fw.update_layout(annotations=[])
    fw.add_trace(go.Scattergl(      # trace 0: baseline reference at 1.0 — never changes
        x=channels, y=np.ones(len(channels)),
        name='Baseline', legendgroup='bg',
        line=dict(color='#bbb', width=1.2, dash='dash'),
        hoverinfo='skip',
    ))
    fw.add_trace(go.Scattergl(      # trace 1: selected spectrum — updated by slider
        x=channels, y=Z_norm[idx],
        name='Selected', legendgroup='sel',
        line=dict(color=color, width=1.5),
        hovertemplate='ch=%{x}<br>%{y:.4f}<extra>Selected</extra>',
    ))
    fw.update_layout(
        height=290, template='plotly_white', hovermode='x unified',
        margin=dict(l=65, r=20, t=38, b=50),
        xaxis=dict(title='Channel'),
        yaxis=dict(title='I / baseline'),
        legend=dict(x=0.01, y=0.99, font_size=9,
                    bgcolor='rgba(255,255,255,0.8)', bordercolor='#ccc', borderwidth=1),
        title=dict(text='…', font_size=11),
    )


# ── Slider callback ────────────────────────────────────────────────────────────

def _sv_on_slider(change):
    if not _sv:
        return
    idx    = change['new']
    ts     = _sv['times'][idx]
    t0, t1 = _sv['t0'], _sv['t1']

    color   = _sv_spectrum_color(ts, t0, t1)
    gas_str = _sv_gas_str(_sv.get('gas_df'), ts, _sv['inst'])
    where   = 'EVENT' if t0 <= ts <= t1 else ('pre' if ts < t0 else 'post')

    with _sv_fw_sel.batch_update():
        _sv_fw_sel.data[1].y         = _sv['Z_norm'][idx]
        _sv_fw_sel.data[1].line      = dict(color=color, width=1.5)
        _sv_fw_sel.layout.title.text = (
            f'{where}  t = {ts.strftime("%H:%M:%S UTC")}  |  {gas_str}')

    with _sv_fw_gas.batch_update():
        _sv_fw_gas.layout.shapes = [
            dict(type='rect', x0=t0, x1=t1, y0=0, y1=1, xref='x', yref='paper',
                 fillcolor='rgba(255,165,0,0.15)', layer='below', line=dict(width=0)),
            dict(type='line', x0=ts.isoformat(), x1=ts.isoformat(),
                 y0=0, y1=1, xref='x', yref='paper',
                 line=dict(color='#222', width=1.5, dash='dot')),
        ]

    region_color = {'EVENT': '#c0392b', 'pre': '#555', 'post': '#555'}[where]
    sv_time_lbl.value = (
        f'<span style="font-size:11px;color:{region_color};">'
        f'<b>{where}</b>  {ts.strftime("%H:%M:%S UTC")}  '
        f'({idx + 1} / {len(_sv["times"])})</span>'
    )

sv_slider.observe(_sv_on_slider, names='value')


# ── Load callback ──────────────────────────────────────────────────────────────

def _sv_on_load(_):
    row  = _PERIODS.iloc[sv_period_dd.value]
    inst = sv_inst_dd.value
    ctx  = sv_ctx_slider.value
    t0   = row.dt_start
    t1   = row.dt_end
    t_lo = t0 - pd.Timedelta(minutes=ctx)
    t_hi = t1 + pd.Timedelta(minutes=ctx)

    sv_status.value = f'<span style="color:#e67e00;font-size:11px;">Loading {inst}…</span>'

    spec_df = load_spectra(inst, t_lo, t_hi)
    gas_df  = load_gas(t_lo, t_hi)

    if spec_df is None:
        sv_status.value = (
            f'<span style="color:red;font-size:11px;">'
            f'No {inst} spectra found for this period.</span>')
        return

    times    = spec_df.index
    Z        = spec_df.values.astype(float)
    channels = np.arange(1, Z.shape[1] + 1)
    ev_mask  = (times >= t0) & (times <= t1)

    sv_status.value = f'<span style="color:#e67e00;font-size:11px;">Fitting baselines…</span>'
    Z_norm = _sv_baseline_correct(Z)

    ev_indices = np.where(ev_mask)[0]
    init_idx   = int(ev_indices[0]) if ev_indices.size else 0

    _sv.update({'gas_df': gas_df, 't0': t0, 't1': t1, 'inst': inst,
                'times': times, 'Z_norm': Z_norm, 'channels': channels})

    _sv_load_gas(_sv_fw_gas, gas_df, t0, t1, inst, ctx)
    _sv_load_sel(_sv_fw_sel, Z_norm, channels, init_idx, t0, t1, times)

    sv_slider.max   = len(times) - 1
    sv_slider.value = init_idx

    sv_status.value = (
        f'<span style="color:#27ae60;font-size:11px;">'
        f'Loaded {len(spec_df)} spectra  ({ev_indices.size} in event window).  '
        f'Drag slider to scrub.</span>'
    )

sv_load_btn.on_click(_sv_on_load)


# ── Layout ────────────────────────────────────────────────────────────────────
sv_ctrl = widgets.VBox([
    widgets.HBox([sv_period_dd, sv_inst_dd]),
    widgets.HBox([sv_ctx_slider, sv_load_btn]),
    sv_status,
], layout=widgets.Layout(
    border='1px solid #ccc', padding='10px 14px',
    background_color='#fafafa', margin='0 0 12px 0',
))
sv_scrub = widgets.VBox([sv_slider, sv_time_lbl],
                        layout=widgets.Layout(margin='6px 0 6px 0'))

display(sv_ctrl, _sv_fw_gas, sv_scrub, _sv_fw_sel)

In [ ]:
from plotly.subplots import make_subplots

# ── Configuration ──────────────────────────────────────────────────────────────
PERIOD_A = 0   # index into _PERIODS (see list printed in cell 3)
PERIOD_B = 3   # index into _PERIODS
INST     = 'Ultra321'
CTX_MIN  = 3   # minutes of pre-event context used as background

# ── Load ───────────────────────────────────────────────────────────────────────
def _load_period(idx):
    row  = _PERIODS.iloc[idx]
    t0, t1 = row.dt_start, row.dt_end
    t_lo = t0 - pd.Timedelta(minutes=CTX_MIN)
    t_hi = t1 + pd.Timedelta(minutes=CTX_MIN)
    spec = load_spectra(INST, t_lo, t_hi)
    return spec, t0, t1, row.description

spec_a, t0a, t1a, desc_a = _load_period(PERIOD_A)
spec_b, t0b, t1b, desc_b = _load_period(PERIOD_B)

# ── Compute difference spectra ─────────────────────────────────────────────────
def _diff(spec_df, t0, t1):
    if spec_df is None or spec_df.empty:
        return None, None
    times   = spec_df.index
    Z       = spec_df.values.astype(float)
    bg_mask = times < t0
    ev_mask = (times >= t0) & (times <= t1)
    if not bg_mask.any() or not ev_mask.any():
        return None, None
    bg  = Z[bg_mask].mean(axis=0)
    ev  = Z[ev_mask].mean(axis=0)
    den = np.where(np.abs(bg) < 1.0, 1.0, np.abs(bg))
    return (ev - bg) / den * 100.0, np.arange(1, Z.shape[1] + 1)

diff_a, ch_a = _diff(spec_a, t0a, t1a)
diff_b, ch_b = _diff(spec_b, t0b, t1b)

# ── Plot ───────────────────────────────────────────────────────────────────────
if diff_a is None and diff_b is None:
    print('No data for either period.')
else:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=[
                            f'Period A (idx={PERIOD_A}): {desc_a}',
                            f'Period B (idx={PERIOD_B}): {desc_b}',
                        ])

    def _add_diff(fig, row, diff, channels, color):
        if diff is None:
            return
        fig.add_trace(go.Scattergl(
            x=channels, y=diff,
            line=dict(color=color, width=0.8),
            fill='tozeroy', fillcolor=color.replace(')', ',0.12)').replace('rgb', 'rgba'),
            name='Δ%',
            hovertemplate='ch=%{x}<br>Δ%%=%{y:.3f}<extra></extra>',
            showlegend=False,
        ), row=row, col=1)
        fig.add_hline(y=0, line=dict(color='#555', width=0.8, dash='dash'),
                      row=row, col=1)

    _add_diff(fig, 1, diff_a, ch_a, 'rgb(225,87,89)')
    _add_diff(fig, 2, diff_b, ch_b, 'rgb(78,121,167)')

    for r in (1, 2):
        fig.update_yaxes(title_text='Δ% from bg', title_font_size=10, row=r, col=1)
    fig.update_xaxes(title_text='Channel', title_font_size=10, row=2, col=1)

    fig.update_layout(
        height=480, template='plotly_white',
        margin=dict(l=65, r=20, t=60, b=50),
        title=dict(text=f'{INST} — difference spectrum comparison (bg = {CTX_MIN} min pre-event)',
                   font_size=12),
    )
    fig.show()